# P2P File Sharing Project — Test Runner

This notebook runs **end-to-end tests** for the BitTorrent-style P2P file sharing system.

**Tests performed:**
1. **Module Import Verification** — Confirm all Python modules load correctly
2. **Unit Tests** — Config parsing, file manager, bitfield, handshake, message encoding
3. **Test 1: Basic 3-Peer Test** — Small synthetic file, 3 peers on localhost
4. **Test 2: Small Config (9 Peers)** — Uses `project_config_file_small.zip` (~2MB, 2 seeds)
5. **Test 3: Large Config (6 Peers)** — Uses `project_config_file_large.zip` (~24MB, 1 seed)
6. **Log Verification** — Validates all required log entries per the project spec
7. **Summary** — Aggregated pass/fail results

In [1]:
############################
# Section 1: Setup & Imports
############################
import os
import sys
import time
import struct
import shutil
import hashlib
import subprocess
import zipfile
import math
import random

# Ensure project root is on the path
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Change to project root
os.chdir(PROJECT_ROOT)

print(f"{'='*60}")
print(f"  P2P FILE SHARING PROJECT — TEST RUNNER")
print(f"{'='*60}")
print(f"  Python version : {sys.version.split()[0]}")
print(f"  Working dir    : {os.getcwd()}")
print()

# Import project modules
from common_config import CommonConfig
from peer_info import PeerInfoEntry
from file_manager import FileManager
from peer_logger import PeerLogger
from connection_handler import ConnectionHandler, HANDSHAKE_HEADER
from connection_handler import CHOKE, UNCHOKE, INTERESTED, NOT_INTERESTED, HAVE, BITFIELD, REQUEST, PIECE

print("  All project modules imported successfully")
print()

# Test results tracker
test_results = []

def record(name, passed, detail=""):
    """Record a test result.  record("test name", True/False, "optional detail")"""
    status = "PASS" if passed else "FAIL"
    test_results.append((name, passed))
    print(f"  [{status}] {name}" + (f" -- {detail}" if detail else ""))

def section_header(section_id, title=""):
    """Print a section header.  section_header("2a", "CommonConfig") or section_header("title")"""
    label = f"{section_id}: {title}" if title else section_id
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")

  P2P FILE SHARING PROJECT — TEST RUNNER
  Python version : 3.12.5
  Working dir    : c:\Users\maksy\Music\Github\NetworksProject

  All project modules imported successfully



## Section 2: Unit Tests — Config Parsing, File Manager, Protocol Messages

In [2]:
###############################################
# 2a: Test CommonConfig parsing
###############################################
section_header("2a", "CommonConfig Parsing")

# Create a temporary Common.cfg
test_cfg = "NumberOfPreferredNeighbors 3\nUnchokingInterval 5\nOptimisticUnchokingInterval 10\nFileName thefile\nFileSize 2167705\nPieceSize 16384\n"
with open("Common.cfg", "w") as f:
    f.write(test_cfg)

cfg = CommonConfig.load("Common.cfg")

record("NumberOfPreferredNeighbors == 3", cfg.number_of_preferred_neighbors == 3, f"got {cfg.number_of_preferred_neighbors}")
record("UnchokingInterval == 5", cfg.unchoking_interval == 5, f"got {cfg.unchoking_interval}")
record("OptimisticUnchokingInterval == 10", cfg.optimistic_unchoking_interval == 10, f"got {cfg.optimistic_unchoking_interval}")
record("FileName == 'thefile'", cfg.file_name == "thefile", f"got '{cfg.file_name}'")
record("FileSize == 2167705", cfg.file_size == 2167705, f"got {cfg.file_size}")
record("PieceSize == 16384", cfg.piece_size == 16384, f"got {cfg.piece_size}")

expected_pieces = math.ceil(2167705 / 16384)
record(f"NumberOfPieces == {expected_pieces}", cfg.get_number_of_pieces() == expected_pieces, f"got {cfg.get_number_of_pieces()}")

os.remove("Common.cfg")


  2a: CommonConfig Parsing
  [PASS] NumberOfPreferredNeighbors == 3 -- got 3
  [PASS] UnchokingInterval == 5 -- got 5
  [PASS] OptimisticUnchokingInterval == 10 -- got 10
  [PASS] FileName == 'thefile' -- got 'thefile'
  [PASS] FileSize == 2167705 -- got 2167705
  [PASS] PieceSize == 16384 -- got 16384
  [PASS] NumberOfPieces == 133 -- got 133


In [3]:
###############################################
# 2b: Test PeerInfo parsing
###############################################
section_header("2b", "PeerInfo Parsing")

peer_cfg = "1001 lin114-00.cise.ufl.edu 6001 1\n1002 lin114-01.cise.ufl.edu 6001 0\n1003 lin114-02.cise.ufl.edu 6001 0\n"
with open("PeerInfo.cfg", "w") as f:
    f.write(peer_cfg)

peers = PeerInfoEntry.load_all("PeerInfo.cfg")

record("Parsed 3 peers", len(peers) == 3, f"got {len(peers)}")
record("Peer 1001 ID correct", peers[0].peer_id == 1001)
record("Peer 1001 host correct", peers[0].host_name == "lin114-00.cise.ufl.edu")
record("Peer 1001 port correct", peers[0].port == 6001)
record("Peer 1001 has file", peers[0].has_file == True)
record("Peer 1002 does NOT have file", peers[1].has_file == False)
record("Peer 1003 does NOT have file", peers[2].has_file == False)

os.remove("PeerInfo.cfg")


  2b: PeerInfo Parsing
  [PASS] Parsed 3 peers -- got 3
  [PASS] Peer 1001 ID correct
  [PASS] Peer 1001 host correct
  [PASS] Peer 1001 port correct
  [PASS] Peer 1001 has file
  [PASS] Peer 1002 does NOT have file
  [PASS] Peer 1003 does NOT have file


In [4]:
###############################################
# 2c: Test FileManager — bitfield, pieces
###############################################
section_header("2c", "FileManager — Bitfield & Piece Management")

# Create a small test config
with open("Common.cfg", "w") as f:
    f.write("NumberOfPreferredNeighbors 2\nUnchokingInterval 2\nOptimisticUnchokingInterval 4\nFileName test.dat\nFileSize 50000\nPieceSize 16384\n")

cfg = CommonConfig.load("Common.cfg")
num_pieces = cfg.get_number_of_pieces()   # ceil(50000/16384) = 4
print(f"  Config: FileSize={cfg.file_size}, PieceSize={cfg.piece_size}, NumPieces={num_pieces}")

# Create a peer dir with the file
os.makedirs("peer_9999", exist_ok=True)
test_data = os.urandom(50000)
with open("peer_9999/test.dat", "wb") as f:
    f.write(test_data)

# Test loading file
fm = FileManager(9999, cfg, has_file=True)
record(f"Num pieces == {num_pieces}", fm.num_pieces == num_pieces)
record("Has all pieces after load", fm.has_all_pieces())
record(f"Pieces count == {num_pieces}", fm.get_pieces_count() == num_pieces)

# Test bitfield
bitfield = fm.get_bitfield()
record(f"Bitfield length == {math.ceil(num_pieces/8)}", len(bitfield) == math.ceil(num_pieces / 8))
# All 4 pieces set: bits 7,6,5,4 of byte 0 = 0b11110000 = 0xF0
record("Bitfield byte[0] == 0xF0 (4 pieces set)", bitfield[0] == 0xF0, f"got 0x{bitfield[0]:02X}")

# Test piece retrieval
piece0 = fm.get_piece(0)
record(f"Piece 0 size == {cfg.piece_size}", len(piece0) == cfg.piece_size)
last_piece_size = 50000 % 16384  # 776
piece_last = fm.get_piece(num_pieces - 1)
record(f"Last piece size == {last_piece_size}", len(piece_last) == last_piece_size, f"got {len(piece_last)}")

# Test FileManager with no file
fm2 = FileManager(9998, cfg, has_file=False)
record("Empty peer has 0 pieces", fm2.get_pieces_count() == 0)
record("Empty peer has_all_pieces == False", not fm2.has_all_pieces())

# Set pieces one by one and verify
for i in range(num_pieces):
    fm2.set_piece(i, fm.get_piece(i))
record("After setting all pieces, has_all_pieces == True", fm2.has_all_pieces())

# Verify saved file matches original
with open("peer_9998/test.dat", "rb") as f:
    saved = f.read()
record("Saved file matches original", saved == test_data, f"len={len(saved)} vs {len(test_data)}")

# Cleanup
shutil.rmtree("peer_9999", ignore_errors=True)
shutil.rmtree("peer_9998", ignore_errors=True)
os.remove("Common.cfg")


  2c: FileManager — Bitfield & Piece Management
  Config: FileSize=50000, PieceSize=16384, NumPieces=4
  [PASS] Num pieces == 4
  [PASS] Has all pieces after load
  [PASS] Pieces count == 4
  [PASS] Bitfield length == 1
  [PASS] Bitfield byte[0] == 0xF0 (4 pieces set) -- got 0xF0
  [PASS] Piece 0 size == 16384
  [PASS] Last piece size == 848 -- got 848
  [PASS] Empty peer has 0 pieces
  [PASS] Empty peer has_all_pieces == False
  [PASS] After setting all pieces, has_all_pieces == True
  [PASS] Saved file matches original -- len=50000 vs 50000


In [5]:
###############################################
# 2d: Test Protocol Message Encoding
###############################################
section_header("2d", "Protocol Message Encoding")

# Test handshake format
print("  Handshake header:", HANDSHAKE_HEADER)
record("Handshake header is 18 bytes", len(HANDSHAKE_HEADER) == 18)
record("Header == b'P2PFILESHARINGPROJ'", HANDSHAKE_HEADER == b"P2PFILESHARINGPROJ")

# Test message type constants
record("CHOKE == 0", CHOKE == 0)
record("UNCHOKE == 1", UNCHOKE == 1)
record("INTERESTED == 2", INTERESTED == 2)
record("NOT_INTERESTED == 3", NOT_INTERESTED == 3)
record("HAVE == 4", HAVE == 4)
record("BITFIELD == 5", BITFIELD == 5)
record("REQUEST == 6", REQUEST == 6)
record("PIECE == 7", PIECE == 7)

# Test handshake construction
handshake = bytearray(32)
handshake[0:18] = HANDSHAKE_HEADER
peer_id = 1001
struct.pack_into("!I", handshake, 28, peer_id)
record("Handshake is 32 bytes", len(handshake) == 32)
record("Handshake bytes 18-27 are zero", all(b == 0 for b in handshake[18:28]))
decoded_id = struct.unpack("!I", handshake[28:32])[0]
record(f"Decoded peer ID == {peer_id}", decoded_id == peer_id)

# Test message framing (length + type + payload)
piece_index = 42
payload = struct.pack("!I", piece_index)
header = struct.pack("!IB", 1 + len(payload), HAVE)
full_msg = header + payload
msg_len = struct.unpack("!I", full_msg[0:4])[0]
msg_type = full_msg[4]
msg_payload = full_msg[5:]
record("HAVE message length == 5", msg_len == 5)
record("HAVE message type == 4", msg_type == HAVE)
decoded_piece = struct.unpack("!I", msg_payload)[0]
record(f"HAVE payload piece index == {piece_index}", decoded_piece == piece_index)


  2d: Protocol Message Encoding
  Handshake header: b'P2PFILESHARINGPROJ'
  [PASS] Handshake header is 18 bytes
  [PASS] Header == b'P2PFILESHARINGPROJ'
  [PASS] CHOKE == 0
  [PASS] UNCHOKE == 1
  [PASS] INTERESTED == 2
  [PASS] NOT_INTERESTED == 3
  [PASS] HAVE == 4
  [PASS] BITFIELD == 5
  [PASS] REQUEST == 6
  [PASS] PIECE == 7
  [PASS] Handshake is 32 bytes
  [PASS] Handshake bytes 18-27 are zero
  [PASS] Decoded peer ID == 1001
  [PASS] HAVE message length == 5
  [PASS] HAVE message type == 4
  [PASS] HAVE payload piece index == 42


## Section 3: Integration Tests — Full Peer-to-Peer File Transfer

The following cells run actual multi-peer file transfer tests using the `peerProcess.py` program.
Each test:
1. Creates `Common.cfg` and `PeerInfo.cfg`
2. Sets up `peer_[ID]/` directories with seed files
3. Launches all peers as separate processes
4. Waits for completion
5. Verifies all peers received identical files (SHA-256 hash comparison)
6. Checks log files for required protocol events
7. Cleans up

In [6]:
###############################################
# Integration Test Helpers
###############################################

import subprocess, hashlib, shutil, zipfile, glob, signal

PROJECT_DIR = os.path.abspath(".")

def write_common_cfg(num_pref, unchoke_int, opt_int, fname, fsize, piece_size, dest="."):
    """Write a Common.cfg file."""
    path = os.path.join(dest, "Common.cfg")
    with open(path, "w") as f:
        f.write(f"NumberOfPreferredNeighbors {num_pref}\n")
        f.write(f"UnchokingInterval {unchoke_int}\n")
        f.write(f"OptimisticUnchokingInterval {opt_int}\n")
        f.write(f"FileName {fname}\n")
        f.write(f"FileSize {fsize}\n")
        f.write(f"PieceSize {piece_size}\n")
    return path

def write_peer_info_cfg(peers, dest="."):
    """Write PeerInfo.cfg.  peers = [(id, host, port, has_file), ...]"""
    path = os.path.join(dest, "PeerInfo.cfg")
    with open(path, "w") as f:
        for pid, host, port, has in peers:
            f.write(f"{pid} {host} {port} {has}\n")
    return path

def make_peer_dir(peer_id, seed_file=None):
    """Create peer_<id>/ directory.  If seed_file, copy it in."""
    d = os.path.join(PROJECT_DIR, f"peer_{peer_id}")
    os.makedirs(d, exist_ok=True)
    if seed_file:
        shutil.copy2(seed_file, d)
    return d

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest().upper()

def launch_peer(peer_id):
    """Launch peerProcess.py as a subprocess; return Popen handle."""
    p = subprocess.Popen(
        [sys.executable, "peerProcess.py", str(peer_id)],
        cwd=PROJECT_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    return p

def wait_peers(procs, timeout=120):
    """Wait for all peer subprocesses to finish, with timeout."""
    deadline = time.time() + timeout
    for pid, p in procs.items():
        remaining = max(0, deadline - time.time())
        try:
            p.wait(timeout=remaining)
        except subprocess.TimeoutExpired:
            p.kill()
            p.wait()
            print(f"  ⚠ Peer {pid} killed (timeout={timeout}s)")

def cleanup_peers(peer_ids):
    """Remove peer directories and log files created during a test."""
    for pid in peer_ids:
        d = os.path.join(PROJECT_DIR, f"peer_{pid}")
        if os.path.isdir(d):
            shutil.rmtree(d, ignore_errors=True)
    for log in glob.glob(os.path.join(PROJECT_DIR, "log_peer_*.log")):
        os.remove(log)
    for cfg in ["Common.cfg", "PeerInfo.cfg"]:
        p = os.path.join(PROJECT_DIR, cfg)
        if os.path.isfile(p):
            os.remove(p)

print("✅ Integration helpers defined — launch_peer(), wait_peers(), cleanup_peers(), etc.")

✅ Integration helpers defined — launch_peer(), wait_peers(), cleanup_peers(), etc.


In [7]:
###############################################
# TEST 3a — Basic 3-peer test (1 seed, 2 leechers, 100 KB file)
###############################################
section_header("3a", "Integration — 3-peer basic transfer (100 KB)")

TEST_FILE = "testfile.dat"
FILE_SIZE = 102400   # 100 KB
PIECE_SIZE = 16384
PEERS = [(1001, "localhost", 6001, 1),
         (1002, "localhost", 6002, 0),
         (1003, "localhost", 6003, 0)]
PEER_IDS = [p[0] for p in PEERS]
procs = {}

try:
    # 1) Cleanup any prior artefacts
    cleanup_peers(PEER_IDS)

    # 2) Write configs
    write_common_cfg(1, 5, 10, TEST_FILE, FILE_SIZE, PIECE_SIZE)
    write_peer_info_cfg(PEERS)
    print(f"  📄 Common.cfg  — File={TEST_FILE}, Size={FILE_SIZE}, PieceSize={PIECE_SIZE}")
    print(f"  📄 PeerInfo.cfg — {len(PEERS)} peers, seed=1001")

    # 3) Generate random seed file & set up directories
    seed_data = os.urandom(FILE_SIZE)
    seed_path = os.path.join(PROJECT_DIR, "peer_1001", TEST_FILE)
    make_peer_dir(1001)
    with open(seed_path, "wb") as f:
        f.write(seed_data)
    seed_hash = sha256_file(seed_path)
    print(f"  🌱 Seed file SHA-256: {seed_hash}")
    for pid in [1002, 1003]:
        make_peer_dir(pid)
    print(f"  📁 Peer directories created: {PEER_IDS}")

    # 4) Launch peers (seed first, then leechers)
    for pid, _, _, _ in PEERS:
        procs[pid] = launch_peer(pid)
        time.sleep(0.5)
    print(f"  🚀 Launched {len(procs)} peer processes")

    # 5) Wait for completion
    TIMEOUT = 90
    print(f"  ⏳ Waiting up to {TIMEOUT}s for all peers to finish …")
    wait_peers(procs, timeout=TIMEOUT)

    # 6) Verify file hashes
    all_match = True
    for pid in PEER_IDS:
        fp = os.path.join(PROJECT_DIR, f"peer_{pid}", TEST_FILE)
        if not os.path.isfile(fp):
            print(f"  ❌ peer_{pid}/{TEST_FILE} MISSING")
            all_match = False
            continue
        h = sha256_file(fp)
        match = h == seed_hash
        status = "✅ match" if match else "❌ MISMATCH"
        print(f"  {status}  peer_{pid}  {h}")
        if not match:
            all_match = False

    record("3a  Basic 3-peer file transfer", all_match)

    # 7) Quick log check
    for pid in PEER_IDS:
        logf = os.path.join(PROJECT_DIR, f"log_peer_{pid}.log")
        exists = os.path.isfile(logf)
        size = os.path.getsize(logf) if exists else 0
        print(f"  📝 log_peer_{pid}.log  exists={exists}  size={size} bytes")

except Exception as ex:
    import traceback; traceback.print_exc()
    record("3a  Basic 3-peer file transfer", False)
finally:
    for p in procs.values():
        if p.poll() is None:
            p.kill(); p.wait()
    cleanup_peers(PEER_IDS)
    time.sleep(2)  # Let OS reclaim ports


  3a: Integration — 3-peer basic transfer (100 KB)
  📄 Common.cfg  — File=testfile.dat, Size=102400, PieceSize=16384
  📄 PeerInfo.cfg — 3 peers, seed=1001
  🌱 Seed file SHA-256: 98CCB62F5016D901FA24085292304A8D90464E32145EFF310612E0F77FC3E6DB
  📁 Peer directories created: [1001, 1002, 1003]
  🚀 Launched 3 peer processes
  ⏳ Waiting up to 90s for all peers to finish …
  ✅ match  peer_1001  98CCB62F5016D901FA24085292304A8D90464E32145EFF310612E0F77FC3E6DB
  ✅ match  peer_1002  98CCB62F5016D901FA24085292304A8D90464E32145EFF310612E0F77FC3E6DB
  ✅ match  peer_1003  98CCB62F5016D901FA24085292304A8D90464E32145EFF310612E0F77FC3E6DB
  [PASS] 3a  Basic 3-peer file transfer
  📝 log_peer_1001.log  exists=True  size=1764 bytes
  📝 log_peer_1002.log  exists=True  size=1876 bytes
  📝 log_peer_1003.log  exists=True  size=1650 bytes


In [8]:
###############################################
# TEST 3b — Small config (9 peers, 2 seeds, ~2 MB "thefile")
###############################################
section_header("3b", "Integration — small config (9 peers, 2 seeds, 2.1 MB)")

SMALL_ZIP = os.path.join(PROJECT_DIR, "Project_Files", "project_config_file_small.zip")
BASE_PORT = 7001
procs = {}
peer_ids = []

try:
    # 1) Extract the zip
    tmp = os.path.join(PROJECT_DIR, "_tmp_small")
    if os.path.isdir(tmp):
        shutil.rmtree(tmp)
    with zipfile.ZipFile(SMALL_ZIP) as zf:
        zf.extractall(tmp)

    inner = os.path.join(tmp, "project_config_file_small")

    cfg = CommonConfig.load(os.path.join(inner, "Common.cfg"))
    fname = cfg.file_name
    fsize = cfg.file_size
    piece_size = cfg.piece_size
    print(f"  📄 File={fname}, Size={fsize}, PieceSize={piece_size}, Pieces={cfg.get_number_of_pieces()}")

    # Build peer list — remap to localhost with unique ports
    orig_peers = PeerInfoEntry.load_all(os.path.join(inner, "PeerInfo.cfg"))
    peers = []
    for i, pi in enumerate(orig_peers):
        peers.append((pi.peer_id, "localhost", BASE_PORT + i, 1 if pi.has_file else 0))
    peer_ids = [p[0] for p in peers]
    seeds = [p[0] for p in peers if p[3] == 1]
    print(f"  👥 Peers: {peer_ids}")
    print(f"  🌱 Seeds: {seeds}")

    # 2) Cleanup & set up dirs
    cleanup_peers(peer_ids)
    write_common_cfg(cfg.number_of_preferred_neighbors, cfg.unchoking_interval,
                     cfg.optimistic_unchoking_interval, fname, fsize, piece_size)
    write_peer_info_cfg(peers)

    seed_hash = None
    for pid in peer_ids:
        src_dir = os.path.join(inner, str(pid))
        src_file = os.path.join(src_dir, fname)
        if os.path.isfile(src_file):
            make_peer_dir(pid, seed_file=src_file)
            if seed_hash is None:
                seed_hash = sha256_file(src_file)
        else:
            make_peer_dir(pid)

    print(f"  🔑 Seed hash: {seed_hash}")

    # 3) Launch all peers
    for pid, _, _, _ in peers:
        procs[pid] = launch_peer(pid)
        time.sleep(0.5)
    print(f"  🚀 Launched {len(procs)} peers")

    # 4) Wait
    TIMEOUT = 120
    print(f"  ⏳ Waiting up to {TIMEOUT}s …")
    wait_peers(procs, timeout=TIMEOUT)

    # 5) Verify
    all_match = True
    for pid in peer_ids:
        fp = os.path.join(PROJECT_DIR, f"peer_{pid}", fname)
        if not os.path.isfile(fp):
            print(f"  ❌ peer_{pid}/{fname} MISSING")
            all_match = False
            continue
        h = sha256_file(fp)
        match = h == seed_hash
        status = "✅" if match else "❌"
        print(f"  {status}  peer_{pid}  {h}")
        if not match:
            all_match = False

    record("3b  Small config 9-peer transfer (2.1 MB)", all_match)

except Exception as ex:
    import traceback; traceback.print_exc()
    record("3b  Small config 9-peer transfer (2.1 MB)", False)
finally:
    for p in procs.values():
        if p.poll() is None:
            p.kill(); p.wait()
    cleanup_peers(peer_ids)
    if os.path.isdir(tmp):
        shutil.rmtree(tmp, ignore_errors=True)
    time.sleep(2)  # Let OS reclaim ports


  3b: Integration — small config (9 peers, 2 seeds, 2.1 MB)
  📄 File=thefile, Size=2167705, PieceSize=16384, Pieces=133
  👥 Peers: [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009]
  🌱 Seeds: [1001, 1006]
  🔑 Seed hash: D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  🚀 Launched 9 peers
  ⏳ Waiting up to 120s …
  ✅  peer_1001  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1002  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1003  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1004  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1005  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1006  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1007  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1008  D7C1822574DFC87DC4A70E47BBEEFC3F15A9EF5B902794084B06DF507713E5DC
  ✅  peer_1009 

In [9]:
###############################################
# TEST 3c — Large config (5 peers, 1 seed, ~24 MB "tree.jpg")
###############################################
section_header("3c", "Integration — large config (5 peers, 1 seed, 24 MB)")

LARGE_ZIP = os.path.join(PROJECT_DIR, "Project_Files", "project_config_file_large.zip")
BASE_PORT = 8001
procs = {}
peer_ids = []

try:
    # 1) Extract zip
    tmp = os.path.join(PROJECT_DIR, "_tmp_large")
    if os.path.isdir(tmp):
        shutil.rmtree(tmp)
    with zipfile.ZipFile(LARGE_ZIP) as zf:
        zf.extractall(tmp)

    inner = os.path.join(tmp, "project_config_file_large")

    cfg = CommonConfig.load(os.path.join(inner, "Common.cfg"))
    fname = cfg.file_name
    fsize = cfg.file_size
    piece_size = cfg.piece_size
    print(f"  📄 File={fname}, Size={fsize}, PieceSize={piece_size}, Pieces={cfg.get_number_of_pieces()}")

    orig_peers = PeerInfoEntry.load_all(os.path.join(inner, "PeerInfo.cfg"))
    peers = []
    for i, pi in enumerate(orig_peers):
        peers.append((pi.peer_id, "localhost", BASE_PORT + i, 1 if pi.has_file else 0))
    peer_ids = [p[0] for p in peers]
    seeds = [p[0] for p in peers if p[3] == 1]
    print(f"  👥 Peers: {peer_ids}")
    print(f"  🌱 Seeds: {seeds}")

    # 2) Setup
    cleanup_peers(peer_ids)
    write_common_cfg(cfg.number_of_preferred_neighbors, cfg.unchoking_interval,
                     cfg.optimistic_unchoking_interval, fname, fsize, piece_size)
    write_peer_info_cfg(peers)

    seed_hash = None
    for pid in peer_ids:
        src_dir = os.path.join(inner, str(pid))
        src_file = os.path.join(src_dir, fname)
        if os.path.isfile(src_file):
            make_peer_dir(pid, seed_file=src_file)
            if seed_hash is None:
                seed_hash = sha256_file(src_file)
        else:
            make_peer_dir(pid)

    print(f"  🔑 Seed hash: {seed_hash}")

    # 3) Launch
    for pid, _, _, _ in peers:
        procs[pid] = launch_peer(pid)
        time.sleep(0.5)
    print(f"  🚀 Launched {len(procs)} peers")

    # 4) Wait — 24 MB needs more time
    TIMEOUT = 300
    print(f"  ⏳ Waiting up to {TIMEOUT}s …")
    wait_peers(procs, timeout=TIMEOUT)

    # 5) Verify
    all_match = True
    for pid in peer_ids:
        fp = os.path.join(PROJECT_DIR, f"peer_{pid}", fname)
        if not os.path.isfile(fp):
            print(f"  ❌ peer_{pid}/{fname} MISSING")
            all_match = False
            continue
        h = sha256_file(fp)
        match = h == seed_hash
        status = "✅" if match else "❌"
        print(f"  {status}  peer_{pid}  {h}")
        if not match:
            all_match = False

    record("3c  Large config 5-peer transfer (24 MB)", all_match)

except Exception as ex:
    import traceback; traceback.print_exc()
    record("3c  Large config 5-peer transfer (24 MB)", False)
finally:
    for p in procs.values():
        if p.poll() is None:
            p.kill(); p.wait()
    cleanup_peers(peer_ids)
    if os.path.isdir(tmp):
        shutil.rmtree(tmp, ignore_errors=True)
    time.sleep(2)  # Let OS reclaim ports


  3c: Integration — large config (5 peers, 1 seed, 24 MB)
  📄 File=tree.jpg, Size=24301474, PieceSize=16384, Pieces=1484
  👥 Peers: [1001, 1002, 1003, 1004, 1005, 1006]
  🌱 Seeds: [1001]
  🔑 Seed hash: 987C7CB3BF013388CC3FE6AA3094EF954B5616E682842ECDFCB85670D8B1087A
  🚀 Launched 6 peers
  ⏳ Waiting up to 300s …
  ✅  peer_1001  987C7CB3BF013388CC3FE6AA3094EF954B5616E682842ECDFCB85670D8B1087A
  ✅  peer_1002  987C7CB3BF013388CC3FE6AA3094EF954B5616E682842ECDFCB85670D8B1087A
  ✅  peer_1003  987C7CB3BF013388CC3FE6AA3094EF954B5616E682842ECDFCB85670D8B1087A
  ✅  peer_1004  987C7CB3BF013388CC3FE6AA3094EF954B5616E682842ECDFCB85670D8B1087A
  ✅  peer_1005  987C7CB3BF013388CC3FE6AA3094EF954B5616E682842ECDFCB85670D8B1087A
  ✅  peer_1006  987C7CB3BF013388CC3FE6AA3094EF954B5616E682842ECDFCB85670D8B1087A
  [PASS] 3c  Large config 5-peer transfer (24 MB)


## Section 4: Log Verification

Runs a quick 3-peer transfer then parses the generated log files to verify all required log entry types per the project specification:
- TCP connection (to / from)
- Preferred neighbors
- Optimistically unchoked neighbor
- Unchoking / choking
- Received 'have', 'interested', 'not interested'
- Downloaded piece (with count)
- Download complete

In [10]:
###############################################
# TEST 4 — Log file verification
###############################################
section_header("4", "Log file content verification")

import re

TEST_FILE = "logtest.dat"
FILE_SIZE = 65536
PIECE_SIZE = 16384
PEERS = [(2001, "localhost", 5001, 1),
         (2002, "localhost", 5002, 0),
         (2003, "localhost", 5003, 0)]
PEER_IDS = [p[0] for p in PEERS]
procs = {}

try:
    cleanup_peers(PEER_IDS)
    time.sleep(1)
    write_common_cfg(1, 5, 10, TEST_FILE, FILE_SIZE, PIECE_SIZE)
    write_peer_info_cfg(PEERS)

    seed_data = os.urandom(FILE_SIZE)
    make_peer_dir(2001)
    with open(os.path.join(PROJECT_DIR, "peer_2001", TEST_FILE), "wb") as f:
        f.write(seed_data)
    for pid in [2002, 2003]:
        make_peer_dir(pid)

    for pid, _, _, _ in PEERS:
        procs[pid] = launch_peer(pid)
        time.sleep(0.5)
    print("  🚀 Launched 3 peers for log test")

    wait_peers(procs, timeout=60)

    for pid in PEER_IDS:
        p = procs[pid]
        rc = p.returncode
        err = p.stderr.read().decode().strip()
        status = "exited(%d)" % rc if rc is not None else "still running"
        print(f"  Peer {pid}: {status}")
        if err:
            for line in err.split('\n')[:3]:
                print(f"    stderr: {line}")
    print()

    # Patterns that MUST appear across ALL logs combined (not necessarily every peer)
    GLOBAL_REQUIRED = {
        "TCP connection (makes)":         r"makes a connection to Peer",
        "TCP connection (connected from)": r"is connected from Peer",
        "Preferred neighbors":            r"has the preferred neighbors",
        "Unchoked by":                    r"is unchoked by",
        "Received 'have'":               r"received the 'have' message",
        "Received 'interested'":         r"received the 'interested' message",
        "Downloaded piece":              r"has downloaded the piece",
        "Download complete":             r"has downloaded the complete file",
    }

    # Aggregate all log content
    all_content = ""
    all_logs_ok = True
    for pid in PEER_IDS:
        logf = os.path.join(PROJECT_DIR, f"log_peer_{pid}.log")
        if not os.path.isfile(logf):
            print(f"  ❌ log_peer_{pid}.log not found!")
            all_logs_ok = False
            continue

        with open(logf, "r") as f:
            content = f.read()
        all_content += content

        lines = [l for l in content.strip().split("\n") if l.strip()]
        ts_pattern = r"\[\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\]"
        ts_count = len(re.findall(ts_pattern, content))
        ts_ok = ts_count == len(lines)
        status_icon = "✅" if ts_ok else "⚠"
        print(f"  📋 log_peer_{pid}.log — {len(lines)} entries, {status_icon} timestamp format")
        if not ts_ok:
            all_logs_ok = False

    print()
    print("  Global log pattern check (across all peer logs):")
    for label, pattern in GLOBAL_REQUIRED.items():
        count = len(re.findall(pattern, all_content))
        if count > 0:
            print(f"    ✅ {label}: {count} total occurrence(s)")
        else:
            print(f"    ❌ {label}: MISSING from all logs")
            all_logs_ok = False

    # Optional patterns
    for label, pattern in [("Optimistically unchoked", r"optimistically unchoked neighbor"),
                           ("Choked by", r"is choked by"),
                           ("Received 'not interested'", r"received the 'not interested' message")]:
        count = len(re.findall(pattern, all_content))
        if count > 0:
            print(f"    ✅ {label}: {count} total occurrence(s)")
        else:
            print(f"    ⚠ {label}: not found (optional)")

    record("4   Log file verification", all_logs_ok)

except Exception as ex:
    import traceback; traceback.print_exc()
    record("4   Log file verification", False)
finally:
    for p in procs.values():
        if p.poll() is None:
            p.kill(); p.wait()
    cleanup_peers(PEER_IDS)
    time.sleep(2)


  4: Log file content verification
  🚀 Launched 3 peers for log test
  Peer 2001: exited(0)
  Peer 2002: exited(0)
  Peer 2003: exited(0)

  📋 log_peer_2001.log — 15 entries, ✅ timestamp format
  📋 log_peer_2002.log — 12 entries, ✅ timestamp format
  📋 log_peer_2003.log — 15 entries, ✅ timestamp format

  Global log pattern check (across all peer logs):
    ✅ TCP connection (makes): 3 total occurrence(s)
    ✅ TCP connection (connected from): 3 total occurrence(s)
    ✅ Preferred neighbors: 2 total occurrence(s)
    ✅ Unchoked by: 2 total occurrence(s)
    ✅ Received 'have': 16 total occurrence(s)
    ✅ Received 'interested': 3 total occurrence(s)
    ✅ Downloaded piece: 8 total occurrence(s)
    ✅ Download complete: 2 total occurrence(s)
    ⚠ Optimistically unchoked: not found (optional)
    ⚠ Choked by: not found (optional)
    ✅ Received 'not interested': 3 total occurrence(s)
  [PASS] 4   Log file verification


## Section 5: Test Summary

In [11]:
###############################################
# FINAL SUMMARY
###############################################
print("=" * 60)
print("  TEST RESULTS SUMMARY")
print("=" * 60)

passed = 0
failed = 0
for name, ok in test_results:
    icon = "✅ PASS" if ok else "❌ FAIL"
    print(f"  {icon}  {name}")
    if ok:
        passed += 1
    else:
        failed += 1

print("-" * 60)
total = passed + failed
print(f"  Total: {total}   Passed: {passed}   Failed: {failed}")
if failed == 0:
    print("\n  🎉  ALL TESTS PASSED — project is fully functional!")
else:
    print(f"\n  ⚠  {failed} test(s) failed — review output above for details.")
print("=" * 60)

  TEST RESULTS SUMMARY
  ✅ PASS  NumberOfPreferredNeighbors == 3
  ✅ PASS  UnchokingInterval == 5
  ✅ PASS  OptimisticUnchokingInterval == 10
  ✅ PASS  FileName == 'thefile'
  ✅ PASS  FileSize == 2167705
  ✅ PASS  PieceSize == 16384
  ✅ PASS  NumberOfPieces == 133
  ✅ PASS  Parsed 3 peers
  ✅ PASS  Peer 1001 ID correct
  ✅ PASS  Peer 1001 host correct
  ✅ PASS  Peer 1001 port correct
  ✅ PASS  Peer 1001 has file
  ✅ PASS  Peer 1002 does NOT have file
  ✅ PASS  Peer 1003 does NOT have file
  ✅ PASS  Num pieces == 4
  ✅ PASS  Has all pieces after load
  ✅ PASS  Pieces count == 4
  ✅ PASS  Bitfield length == 1
  ✅ PASS  Bitfield byte[0] == 0xF0 (4 pieces set)
  ✅ PASS  Piece 0 size == 16384
  ✅ PASS  Last piece size == 848
  ✅ PASS  Empty peer has 0 pieces
  ✅ PASS  Empty peer has_all_pieces == False
  ✅ PASS  After setting all pieces, has_all_pieces == True
  ✅ PASS  Saved file matches original
  ✅ PASS  Handshake header is 18 bytes
  ✅ PASS  Header == b'P2PFILESHARINGPROJ'
  ✅ PASS  CHO